In [1]:
import numpy as np
import statsmodels.api as sm
from joblib import Parallel, delayed

In [2]:
# --------------------------------------------------
# 1. Stratified subsampling on y
# --------------------------------------------------
def stratified_subsample(
    X,
    y,
    n_strata=30,
    max_per_stratum=8000,
    random_state=0
):
    """
    Stratified random subsample based on y using quantile bins.

    With your resources, n_strata=30 and max_per_stratum=8000
    gives up to ~240k rows, which is a good sweet spot.

    Returns:
        X_sub, y_sub, idx_sub
    """
    rng = np.random.default_rng(random_state)
    y = np.asarray(y).ravel()

    # quantile-based bin edges for y
    quantiles = np.linspace(0, 1, n_strata + 1)
    bin_edges = np.quantile(y, quantiles)
    bin_edges = np.unique(bin_edges)

    if len(bin_edges) - 1 < 1:
        raise ValueError("Not enough unique values in y to build strata.")

    strata_indices = []

    for i in range(len(bin_edges) - 1):
        left = bin_edges[i]
        right = bin_edges[i + 1]

        if i == len(bin_edges) - 2:
            mask = (y >= left) & (y <= right)
        else:
            mask = (y >= left) & (y < right)

        idx = np.where(mask)[0]
        if len(idx) == 0:
            continue

        if len(idx) > max_per_stratum:
            idx = rng.choice(idx, size=max_per_stratum, replace=False)

        strata_indices.append(idx)

    if len(strata_indices) == 0:
        raise ValueError("No samples were selected. Check your strata settings.")

    idx_sub = np.concatenate(strata_indices)
    return X[idx_sub], y[idx_sub], idx_sub


# --------------------------------------------------
# 2. Quantile regression for a single tau
# --------------------------------------------------
def fit_quantile_regression(X, y, tau):
    """
    Fit linear quantile regression for a given tau using statsmodels.
    Returns (tau, params, result_object).
    """
    X = np.asarray(X)
    y = np.asarray(y).ravel()

    # add intercept
    X_const = sm.add_constant(X, has_constant="add")

    model = sm.QuantReg(y, X_const)
    res = model.fit(q=tau)
    return tau, res.params, res


# --------------------------------------------------
# 3. Fit many quantiles in parallel
# --------------------------------------------------
def fit_many_quantiles_parallel(
    X,
    y,
    quantiles=None,
    n_jobs=None,
    n_strata=30,
    max_per_stratum=8000,
    random_state=0,
):
    """
    Stratify & subsample, then fit multiple quantiles in parallel.

    With your hardware:
        - n_jobs can be up to 64, but we cap at len(quantiles).
        - n_strata=30, max_per_stratum=8000 -> up to ~240k rows.
    """
    if quantiles is None:
        # 0.05, 0.10, 0.15, ..., 0.95
        quantiles = np.arange(0.05, 1.00, 0.05)

    if n_jobs is None:
        n_jobs = min(64, len(quantiles))  # avoid spawning more jobs than needed

    # 1) stratified subsampling
    X_sub, y_sub, idx_sub = stratified_subsample(
        X,
        y,
        n_strata=n_strata,
        max_per_stratum=max_per_stratum,
        random_state=random_state,
    )

    # 2) parallel quantile fits
    jobs = Parallel(n_jobs=n_jobs)(
        delayed(fit_quantile_regression)(X_sub, y_sub, tau) for tau in quantiles
    )

    params_dict = {tau: params for tau, params, _ in jobs}
    model_dict = {tau: res for tau, _, res in jobs}

    return {
        "quantiles": list(quantiles),
        "params": params_dict,   # tau -> coefficient vector (incl. intercept)
        "models": model_dict,    # tau -> full statsmodels result
        "indices": idx_sub,      # indices of rows used from original X,y
    }

In [4]:
# --------------------------------------------------
# 4. Example usage with your dimensions
# --------------------------------------------------
#if __name__ == "__main__":
# your scale: 90*4769 samples, 1001 predictors
n_samples = 90 * 4769       # 429,210
n_features = 1001

rng = np.random.default_rng(42)
X = rng.normal(size=(n_samples, n_features)).astype(np.float32)

true_beta = rng.normal(size=(n_features,)).astype(np.float32) / np.sqrt(n_features)
y = X @ true_beta + rng.normal(scale=1.0, size=n_samples).astype(np.float32)

quantiles = np.arange(0.05, 1.00, 0.05)  # 0.05, 0.10, ..., 0.95



In [6]:
print(true_beta.shape)

(1001,)


In [7]:
# Test
X_sub, y_sub, idx_sub = stratified_subsample(
        X,
        y,
        n_strata=30,
        max_per_stratum=8000,
        random_state=42,
    )

In [12]:
print("X_sub shape:", X_sub.shape)
print("y_sub shape:", y_sub.shape)
print("idx_sub:", idx_sub.shape)

X_sub shape: (240000, 1001)
y_sub shape: (240000,)
idx_sub: (240000,)


In [14]:
%%time
results = fit_many_quantiles_parallel(
    X,
    y,
    quantiles=quantiles,
    n_jobs=19,          # one job per quantile; fits comfortably under 64 cores
    n_strata=5, #30,
    max_per_stratum=100,
    random_state=42,
)

beta_q05 = results["params"][0.05]
beta_q50 = results["params"][0.5]

print("beta_q05 shape:", beta_q05.shape)
print("beta_q50 shape:", beta_q50.shape)
print("first 5 coefficients at tau=0.5:", beta_q50[:5])

beta_q05 shape: (1002,)
beta_q50 shape: (1002,)
first 5 coefficients at tau=0.5: [ 0.04508735 -0.00319067 -0.05034502  0.06239848 -0.01137334]
CPU times: user 316 ms, sys: 274 ms, total: 590 ms
Wall time: 17.3 s


In [5]:
%%time
results = fit_many_quantiles_parallel(
    X,
    y,
    quantiles=quantiles,
    n_jobs=19,          # one job per quantile; fits comfortably under 64 cores
    n_strata=5, #30,
    max_per_stratum=1000,
    random_state=42,
)

beta_q05 = results["params"][0.05]
beta_q50 = results["params"][0.5]

print("beta_q05 shape:", beta_q05.shape)
print("beta_q50 shape:", beta_q50.shape)
print("first 5 coefficients at tau=0.5:", beta_q50[:5])

Exception ignored in: <function ResourceTracker.__del__ at 0x1485031d5e40>
Traceback (most recent call last):
  File "/home/sc.uni-leipzig.de/fl53wumy/.conda/envs/dpa/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/sc.uni-leipzig.de/fl53wumy/.conda/envs/dpa/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/sc.uni-leipzig.de/fl53wumy/.conda/envs/dpa/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x14b5691c1e40>
Traceback (most recent call last):
  File "/home/sc.uni-leipzig.de/fl53wumy/.conda/envs/dpa/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/sc.uni-leipzig.de/fl53wumy/.conda/envs/dpa/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/sc.uni-leipzig.de/fl53wumy/.conda/envs/dpa/lib/python3.13/multipr

beta_q05 shape: (1002,)
beta_q50 shape: (1002,)
first 5 coefficients at tau=0.5: [ 0.00226371 -0.02022557 -0.04351634  0.00847497 -0.00279128]
CPU times: user 7.96 s, sys: 6.73 s, total: 14.7 s
Wall time: 58min 20s


Exception ignored in: <function ResourceTracker.__del__ at 0x152736175e40>
Traceback (most recent call last):
  File "/home/sc.uni-leipzig.de/fl53wumy/.conda/envs/dpa/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/sc.uni-leipzig.de/fl53wumy/.conda/envs/dpa/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/sc.uni-leipzig.de/fl53wumy/.conda/envs/dpa/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x14574abe1e40>
Traceback (most recent call last):
  File "/home/sc.uni-leipzig.de/fl53wumy/.conda/envs/dpa/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/sc.uni-leipzig.de/fl53wumy/.conda/envs/dpa/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/sc.uni-leipzig.de/fl53wumy/.conda/envs/dpa/lib/python3.13/multipr